In [10]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator 

In [11]:
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini",temperature=0)


In [12]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "rejected"] = Field(description="The evaluation of the tweet")
    feedback: str = Field(description="feedback for the tweet")

In [13]:
structured_tweet_evaluation_model = model.with_structured_output(TweetEvaluation)

In [ ]:
class TwweetState(TypedDict):
    topic: str
    generated_tweet: str
    evaluation: str
    feedback: str
    iteration: int
    max_iterations: int
    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]


In [ ]:
def generate_tweet(state: TwweetState) -> TwweetState:
    prompt = f"""You are a social media manager tasked with creating engaging tweets about {state['topic']}. 
    Generate a tweet that is concise, attention-grabbing, and relevant to the topic. 
    The tweet should be no more than 280 characters long."""
    
    generated_tweet = model.invoke(prompt).content
    return {"generated_tweet": generated_tweet, "tweet_history": [generated_tweet]}

In [ ]:
def evaluate_tweet(state: TwweetState) -> TweetEvaluation:
    prompt = f"""You are a social media manager evaluating the following tweet: {state['generated_tweet']}. 
    The topic is {state['topic']}. 
    Evaluate whether the tweet is engaging, relevant, and concise. 
    Provide an evaluation of "approved" or "rejected" along with feedback for improvement."""
    
    evaluation = structured_tweet_evaluation_model.invoke(prompt)
    return {"evaluation": evaluation.evaluation, "feedback": evaluation.feedback, "feedback_history": [evaluation.feedback]}

In [ ]:
def optimize_tweet(state: TwweetState) -> TwweetState:
    prompt = f"""You are a social media manager tasked with optimizing the following tweet: {state['generated_tweet']}. 
    The topic is {state['topic']}. 
    Based on the feedback provided: {state['feedback']}, optimize the tweet to make it more engaging, relevant, and concise. 
    The optimized tweet should be no more than 280 characters long."""
    
    optimized_tweet = model.invoke(prompt).content
    iteration = state['iteration'] + 1
    return {"generated_tweet": optimized_tweet, "tweet_history": [optimized_tweet], "iteration": iteration}

In [18]:
def route_evaluation(state: TwweetState) -> str:
    if state['evaluation'] == "approved":
        return "approved"
    else:
        return "rejected"

In [21]:
graph=StateGraph(TwweetState)
graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("route_evaluation", route_evaluation)
graph.add_node("optimize_tweet", optimize_tweet)
graph.add_edge(START,"generate_tweet")
graph.add_edge("generate_tweet", "evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet", route_evaluation,{"approved": END, "rejected": "optimize_tweet"})
graph.add_edge("optimize_tweet", "evaluate_tweet")
workflow = graph.compile()

In [22]:
response=workflow.invoke({"topic": "Artificial Intelligence", "iteration": 0, "max_iterations": 5})
print(response)

{'topic': 'Artificial Intelligence', 'generated_tweet': "🤖✨ AI is not just a buzzword; it's transforming our world! From personalized recommendations to groundbreaking medical advancements, the future is here. How do you see AI shaping your life in the next 5 years? Share your thoughts! #ArtificialIntelligence #FutureTech #Innovation", 'evaluation': 'approved', 'feedback': 'The tweet is engaging and relevant, as it addresses a hot topic in technology and invites followers to share their thoughts, fostering interaction. It effectively uses emojis to capture attention and includes relevant hashtags to increase visibility. To improve, consider making it slightly more concise by removing some adjectives, which could enhance clarity and impact.', 'iteration': 0, 'max_iterations': 5}
